# Model Evaluation & Validation — Tutorial

**Primary outcome:** Choose appropriate evaluation metrics for different problem types, implement cross-validation, detect overfitting with learning curves, and tune hyperparameters systematically.

**Time:** 75–90 minutes

---

## What You Will Build

By the end of this tutorial you will have a complete `evaluate_classification_model()` pipeline that prints accuracy, precision, recall, F1, and AUC, and plots a confusion matrix + ROC curve side-by-side — and you will understand *why* each number matters.

## Sections

1. Setup & Data Loading  
2. Why Accuracy Isn't Enough  
3. The Confusion Matrix  
4. Precision, Recall & F1  
5. ROC Curve & AUC  
6. Precision-Recall Curve  
7. Regression Metrics  
8. Cross-Validation  
9. Learning Curves  
10. Validation Curves  
11. Hyperparameter Tuning: Grid Search  
12. Hyperparameter Tuning: Random Search  
13. Putting It All Together  
14. Practice Exercises  

---

## 1. Setup & Data Loading

We import everything up front so there are no surprises later.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    GridSearchCV, RandomizedSearchCV, learning_curve, validation_curve
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score, RocCurveDisplay,
    PrecisionRecallDisplay,
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error
)
from scipy.stats import randint, uniform

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print("All libraries imported successfully.")

### Load the Breast Cancer Dataset

We use this for all classification sections. 569 tumour samples, 30 numeric features.
Target: **0 = malignant**, **1 = benign**.

Notice the class imbalance — 212 malignant vs 357 benign. This will matter when we choose metrics.

In [ ]:
# Load breast cancer dataset
cancer = load_breast_cancer()
X_cancer = cancer.data
y_cancer = cancer.target  # 0 = malignant, 1 = benign

counts = np.bincount(y_cancer)
print("Breast Cancer Dataset")
print(f"  Total samples : {len(y_cancer)}")
print(f"  Malignant (0) : {counts[0]}  ({counts[0]/len(y_cancer)*100:.1f}%)")
print(f"  Benign    (1) : {counts[1]}  ({counts[1]/len(y_cancer)*100:.1f}%)")
print(f"  Features      : {X_cancer.shape[1]}")

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)
print(f"\nTrain size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Fit base Logistic Regression
base_lr = LogisticRegression(max_iter=10000, random_state=42)
base_lr.fit(X_train_sc, y_train)
print(f"\nBase LogisticRegression test accuracy: {base_lr.score(X_test_sc, y_test):.4f}")

---

## 2. Why Accuracy Isn't Enough

**The trap:** A model that always predicts the majority class scores very high accuracy on an imbalanced dataset while being completely useless.

**Real-world consequences:**
- **Cancer screening:** A model that always says "benign" gets ~63 % accuracy on breast cancer data — but misses every malignant tumour.
- **Fraud detection:** Fraud is ~0.1 % of transactions. A model that never flags fraud is 99.9 % accurate but catches nothing.

Let's prove this with a synthetic 95 %/5 % dataset.

In [ ]:
# Build a 95/5 imbalanced dataset
n = 1000
y_imb = np.array([0] * 950 + [1] * 50)
X_imb = np.random.randn(n, 5)

X_imb_tr, X_imb_te, y_imb_tr, y_imb_te = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb
)

# Dummy classifier: always predicts majority class (0)
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_imb_tr, y_imb_tr)
y_dummy_pred = dummy.predict(X_imb_te)

print("=== Dummy 'predict everything as 0' classifier ===")
print(f"Accuracy : {accuracy_score(y_imb_te, y_dummy_pred):.2%}")
print(f"Precision: {precision_score(y_imb_te, y_dummy_pred, zero_division=0):.2%}")
print(f"Recall   : {recall_score(y_imb_te, y_dummy_pred, zero_division=0):.2%}")
print(f"F1       : {f1_score(y_imb_te, y_dummy_pred, zero_division=0):.2%}")
print()
print("Key insight: 95 % accuracy, 0 % recall on the minority class.")
print("In cancer screening this means EVERY positive case is missed.")

---

## 3. The Confusion Matrix

The confusion matrix breaks predictions into four quadrants:

| | Predicted Negative | Predicted Positive |
|---|---|---|
| **Actual Negative** | True Negative (TN) | False Positive (FP) |
| **Actual Positive** | False Negative (FN) | True Positive (TP) |

**Medical context:**
- **FN (missed cancer):** Patient told they are fine but have malignant tumour → worst outcome.
- **FP (false alarm):** Patient told they may have cancer but do not → causes anxiety but caught with follow-up.
- A good cancer screening model prioritises **minimising FN** even at the cost of more FP.

In [ ]:
y_pred_lr = base_lr.predict(X_test_sc)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr,
    display_labels=["Malignant (0)", "Benign (1)"],
    cmap="Blues", ax=ax
)
ax.set_title("Confusion Matrix — Logistic Regression on Breast Cancer", fontsize=12)
plt.tight_layout()
plt.show()

# Manually compute from confusion_matrix()
cm = confusion_matrix(y_test, y_pred_lr)
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (TN): {tn}  — malignant correctly identified")
print(f"False Positives (FP): {fp}  — malignant called benign (missed cancer!)")
print(f"False Negatives (FN): {fn}  — benign called malignant (false alarm)")
print(f"True Positives  (TP): {tp}  — benign correctly identified")

---

## 4. Precision, Recall & F1

Three metrics derived from the confusion matrix cells:

$$\text{Precision} = \frac{TP}{TP + FP} \quad \text{(of all positive predictions, how many were correct?)}$$

$$\text{Recall} = \frac{TP}{TP + FN} \quad \text{(of all actual positives, how many did we catch?)}$$

$$\text{F1} = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} \quad \text{(harmonic mean — rewards balance)}$$

**Precision-Recall tradeoff:** Lowering the decision threshold → more positives predicted → recall goes up, precision goes down. Raising it does the reverse.

In [ ]:
# Manually compute from cm cells
precision_manual = tp / (tp + fp)
recall_manual    = tp / (tp + fn)
f1_manual        = 2 * precision_manual * recall_manual / (precision_manual + recall_manual)

print("Manually computed (positive class = benign/1):")
print(f"  Precision = TP/(TP+FP) = {tp}/({tp}+{fp}) = {precision_manual:.4f}")
print(f"  Recall    = TP/(TP+FN) = {tp}/({tp}+{fn}) = {recall_manual:.4f}")
print(f"  F1        = {f1_manual:.4f}")

print("\nsklearn classification_report:")
print(classification_report(y_test, y_pred_lr,
                            target_names=["Malignant", "Benign"]))

In [ ]:
# Precision and Recall vs decision threshold
y_prob_lr = base_lr.predict_proba(X_test_sc)[:, 1]
thresholds = np.linspace(0, 1, 100)

precisions, recalls, f1s = [], [], []
for t in thresholds:
    y_t = (y_prob_lr >= t).astype(int)
    precisions.append(precision_score(y_test, y_t, zero_division=0))
    recalls.append(recall_score(y_test, y_t, zero_division=0))
    f1s.append(f1_score(y_test, y_t, zero_division=0))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, precisions, label="Precision", color="steelblue")
ax.plot(thresholds, recalls,    label="Recall",    color="tomato")
ax.plot(thresholds, f1s,        label="F1",        color="seagreen", linestyle="--")
ax.axvline(0.5, color="grey", linestyle=":", label="Default threshold (0.5)")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision, Recall & F1 vs Decision Threshold")
ax.legend()
plt.tight_layout()
plt.show()

print("Observation: as the threshold rises, precision increases but recall falls.")
print("For cancer screening you would lower the threshold to catch more true positives.")

---

## 5. ROC Curve & AUC

The **ROC curve** plots True Positive Rate (Recall) against False Positive Rate across all thresholds.

**AUC (Area Under the Curve)** summarises the curve in one number:
- **0.5** → random classifier (diagonal line)
- **1.0** → perfect classifier
- **Interpretation:** AUC = the probability that the model ranks a random positive example higher than a random negative example.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

# Our model
RocCurveDisplay.from_estimator(base_lr, X_test_sc, y_test,
                               name="Logistic Regression", ax=ax)

# Random classifier baseline
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random (AUC = 0.50)")

auc_score = roc_auc_score(y_test, y_prob_lr)
ax.set_title(f"ROC Curve  (AUC = {auc_score:.4f})")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

print(f"AUC = {auc_score:.4f}")
print(f"This means our model ranks a random malignant tumour above a random benign one {auc_score*100:.1f}% of the time.")

---

## 6. Precision-Recall Curve

**When to use PR curve over ROC:**
- Use the PR curve when the **positive class is rare** (imbalanced datasets).
- ROC can look optimistic because it includes TN in the False Positive Rate denominator — a very large TN pool makes FPR small even if FP is significant.
- The PR curve ignores TN entirely, so it directly shows how well the model finds positives.

**Rule of thumb:** Cancer detection, fraud, rare disease → use PR curve. Balanced classes → either works.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC
RocCurveDisplay.from_estimator(base_lr, X_test_sc, y_test,
                               name="Logistic Regression", ax=axes[0])
axes[0].plot([0, 1], [0, 1], "--", color="grey", label="Random")
axes[0].set_title("ROC Curve")
axes[0].legend(loc="lower right")

# Precision-Recall
PrecisionRecallDisplay.from_estimator(base_lr, X_test_sc, y_test,
                                      name="Logistic Regression", ax=axes[1])
axes[1].set_title("Precision-Recall Curve")

plt.suptitle("ROC vs Precision-Recall Curve — Breast Cancer", fontsize=13)
plt.tight_layout()
plt.show()

---

## 7. Regression Metrics

We switch to the **Tips dataset** (seaborn) and predict `total_bill` from other features.

| Metric | Formula | Unit | Intuition |
|--------|---------|------|-----------|
| MAE | mean(|y − ŷ|) | dollars | Average absolute error |
| MSE | mean((y − ŷ)²) | dollars² | Penalises large errors more |
| RMSE | √MSE | dollars | Back to original units |
| R² | 1 − SS_res/SS_tot | unitless | Fraction of variance explained (1 = perfect) |
| MAPE | mean(|y − ŷ|/|y|) | % | Relative error |


In [ ]:
# Load tips dataset
tips = sns.load_dataset('tips')
print(tips.head())
print(f"\nShape: {tips.shape}")

# Feature engineering: encode categoricals
tips_enc = pd.get_dummies(tips, columns=['sex', 'smoker', 'day', 'time'], drop_first=True)

X_tips = tips_enc.drop('total_bill', axis=1).values.astype(float)
y_tips = tips_enc['total_bill'].values

X_tips_tr, X_tips_te, y_tips_tr, y_tips_te = train_test_split(
    X_tips, y_tips, test_size=0.2, random_state=42
)

lr_reg = LinearRegression()
lr_reg.fit(X_tips_tr, y_tips_tr)
y_tips_pred = lr_reg.predict(X_tips_te)

mae  = mean_absolute_error(y_tips_te, y_tips_pred)
mse  = mean_squared_error(y_tips_te, y_tips_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_tips_te, y_tips_pred)
mape = mean_absolute_percentage_error(y_tips_te, y_tips_pred)

print("\nRegression Metrics on Tips Dataset")
print(f"  MAE  = ${mae:.2f}   → on average predictions are off by ${mae:.2f}")
print(f"  MSE  = {mse:.2f}")
print(f"  RMSE = ${rmse:.2f}   → in original dollar units")
print(f"  R²   = {r2:.4f}  → model explains {r2*100:.1f}% of variance in total_bill")
print(f"  MAPE = {mape*100:.1f}%   → predictions are ~{mape*100:.0f}% off relative to actual")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Actual vs Predicted
axes[0].scatter(y_tips_te, y_tips_pred, alpha=0.6, color="steelblue")
mn, mx = y_tips_te.min(), y_tips_te.max()
axes[0].plot([mn, mx], [mn, mx], "--", color="tomato", label="Perfect prediction")
axes[0].set_xlabel("Actual total_bill ($)")
axes[0].set_ylabel("Predicted total_bill ($)")
axes[0].set_title("Actual vs Predicted")
axes[0].legend()

# Residuals
residuals = y_tips_te - y_tips_pred
axes[1].scatter(y_tips_pred, residuals, alpha=0.6, color="seagreen")
axes[1].axhline(0, color="tomato", linestyle="--")
axes[1].set_xlabel("Predicted total_bill ($)")
axes[1].set_ylabel("Residual (actual − predicted)")
axes[1].set_title("Residual Plot")

plt.suptitle("Linear Regression — Tips Dataset", fontsize=13)
plt.tight_layout()
plt.show()

print("A good residual plot shows points randomly scattered around 0 with no pattern.")

---

## 8. Cross-Validation

A single train/test split gives one estimate that may be lucky or unlucky depending on which samples ended up in the test set.

**K-Fold Cross-Validation** solves this by repeating the process K times:
1. Split data into K equal "folds".
2. For each fold: train on the other K−1 folds, evaluate on this fold.
3. Report mean ± std across K scores.

**Stratified K-Fold** preserves the class ratio in every fold — always use this for classification.

In [ ]:
# Back to breast cancer
X_sc_full = scaler.fit_transform(X_cancer)  # scale full dataset for CV

cv_model = LogisticRegression(max_iter=10000, random_state=42)

cv_scores = cross_val_score(
    cv_model, X_sc_full, y_cancer,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc'
)

# Single split score for comparison
cv_model.fit(X_train_sc, y_train)
single_auc = roc_auc_score(y_test, cv_model.predict_proba(X_test_sc)[:, 1])

print("5-Fold Stratified Cross-Validation (ROC-AUC)")
print(f"  Fold scores : {np.round(cv_scores, 4)}")
print(f"  Mean ± Std  : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  95% CI      : [{cv_scores.mean()-2*cv_scores.std():.4f}, {cv_scores.mean()+2*cv_scores.std():.4f}]")
print(f"\n  Single train/test split AUC: {single_auc:.4f}")
print("\n  CV gives a more honest estimate by averaging over multiple test sets.")

In [ ]:
# Visualise: boxplot of 5-fold CV scores vs single split
fig, ax = plt.subplots(figsize=(7, 4))

ax.boxplot(cv_scores, positions=[1], widths=0.4, patch_artist=True,
           boxprops=dict(facecolor="steelblue", alpha=0.6))
ax.scatter([1]*5, cv_scores, color="steelblue", zorder=5, s=60, label="CV fold scores")
ax.axhline(single_auc, color="tomato", linestyle="--", linewidth=2,
           label=f"Single split AUC = {single_auc:.4f}")
ax.set_xlim(0.5, 1.5)
ax.set_xticks([1])
ax.set_xticklabels(["5-Fold CV"])
ax.set_ylabel("ROC-AUC")
ax.set_title("5-Fold CV Scores vs Single Train/Test Split")
ax.legend()
plt.tight_layout()
plt.show()

---

## 9. Learning Curves

A **learning curve** plots training and validation scores as the training set size grows.

**Diagnosing problems:**

| Pattern | Diagnosis | Fix |
|---------|-----------|-----|
| Train score high, val score low (large gap) | **Overfitting** | More data, regularisation, simpler model |
| Both scores low and flat | **Underfitting** | More features, more complex model |
| Scores converge as size grows | Healthy — more data helps | Collect more data |


In [ ]:
lc_model = LogisticRegression(max_iter=10000, random_state=42)

train_sizes, train_scores, val_scores = learning_curve(
    lc_model, X_sc_full, y_cancer,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_mean, 'o-', color='steelblue', label='Training AUC')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.2, color='steelblue')
ax.plot(train_sizes, val_mean, 's-', color='tomato', label='Validation AUC')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                alpha=0.2, color='tomato')
ax.set_xlabel('Training Set Size')
ax.set_ylabel('ROC-AUC')
ax.set_title('Learning Curve — Logistic Regression on Breast Cancer')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

gap = train_mean[-1] - val_mean[-1]
print(f"Final train AUC: {train_mean[-1]:.4f}")
print(f"Final val   AUC: {val_mean[-1]:.4f}")
print(f"Gap (overfitting indicator): {gap:.4f}")
if gap < 0.02:
    print("Small gap — model generalises well.")
else:
    print("Large gap — model is overfitting. Try regularisation or more data.")

---

## 10. Validation Curves

A **validation curve** plots train and validation scores as a *single hyperparameter* varies.

- Too simple (e.g., shallow tree) → underfitting → both scores low.
- Too complex (e.g., very deep tree) → overfitting → train score high, val score drops.
- The optimal value sits where the validation score peaks.

In [ ]:
param_range = np.arange(1, 21)

train_scores_vc, val_scores_vc = validation_curve(
    DecisionTreeClassifier(random_state=42),
    X_cancer, y_cancer,
    param_name='max_depth',
    param_range=param_range,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy',
    n_jobs=-1
)

vc_train_mean = train_scores_vc.mean(axis=1)
vc_val_mean   = val_scores_vc.mean(axis=1)
vc_val_std    = val_scores_vc.std(axis=1)

best_depth = param_range[np.argmax(vc_val_mean)]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(param_range, vc_train_mean, 'o-', color='steelblue', label='Training Accuracy')
ax.plot(param_range, vc_val_mean,   's-', color='tomato',    label='Validation Accuracy')
ax.fill_between(param_range,
                vc_val_mean - vc_val_std,
                vc_val_mean + vc_val_std,
                alpha=0.2, color='tomato')
ax.axvline(best_depth, color='seagreen', linestyle='--',
           label=f'Best max_depth = {best_depth}')
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Validation Curve — DecisionTree max_depth (Breast Cancer)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Best max_depth by validation accuracy: {best_depth}")
print(f"Validation accuracy at best depth    : {vc_val_mean[best_depth-1]:.4f}")
print("For depths > optimal, training accuracy reaches 1.0 (memorisation / overfitting).")

---

## 11. Hyperparameter Tuning: Grid Search

`GridSearchCV` exhaustively evaluates every combination in a parameter grid.

**Use when:** The parameter space is small and you can afford the compute.

We tune `SVC` over:
- `C ∈ {0.01, 0.1, 1, 10}` — regularisation strength
- `kernel ∈ {linear, rbf}` — decision boundary shape

That is 4 × 2 = 8 combinations, each evaluated with 5-fold CV → **40 fits**.

In [ ]:
param_grid = {
    'C':      [0.01, 0.1, 1, 10],
    'kernel': ['linear', 'rbf']
}

grid_search = GridSearchCV(
    SVC(probability=True, random_state=42),
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train_sc, y_train)

print(f"Best parameters : {grid_search.best_params_}")
print(f"Best CV AUC     : {grid_search.best_score_:.4f}")

# Heatmap of mean test scores
results_df = pd.DataFrame(grid_search.cv_results_)
pivot = results_df.pivot_table(
    index='param_C', columns='param_kernel', values='mean_test_score'
)

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, vmin=0.95)
ax.set_title('Grid Search — Mean CV ROC-AUC (SVC)')
ax.set_xlabel('kernel')
ax.set_ylabel('C')
plt.tight_layout()
plt.show()

---

## 12. Hyperparameter Tuning: Random Search

`RandomizedSearchCV` samples randomly from distributions instead of evaluating every combination.

**Advantage:** Much faster for large parameter spaces. With 10 iterations you cover more of a 3D space than Grid Search does with 8 fixed combinations.

**Rule of thumb:** Use Random Search when you have ≥ 3 hyperparameters. Use Grid Search to fine-tune after narrowing the range.

In [ ]:
param_dist = {
    'n_estimators':      randint(50, 300),
    'max_depth':         randint(2, 20),
    'min_samples_split': randint(2, 20),
    'max_features':      ['sqrt', 'log2']
}

rand_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=30,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=0
)
rand_search.fit(X_train_sc, y_train)

print("Random Search — RandomForestClassifier")
print(f"Best parameters: {rand_search.best_params_}")
print(f"Best CV AUC    : {rand_search.best_score_:.4f}")

print("\nComparison:")
print(f"  Grid Search  best AUC (SVC, 8 combos  ): {grid_search.best_score_:.4f}")
print(f"  Random Search best AUC (RF, 30 samples): {rand_search.best_score_:.4f}")
print("  Random Search explored a much larger space with only 30 fits.")

---

## 13. Putting It All Together: Evaluation Pipeline

We write a reusable `evaluate_classification_model()` function and run it on three classifiers.

In [ ]:
def evaluate_classification_model(model, X_train, X_test, y_train, y_test, model_name="Model"):
    """Fit model, print key metrics, and plot confusion matrix + ROC curve."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)

    print(f"{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1        : {f1:.4f}")
    print(f"  ROC-AUC   : {auc:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Confusion Matrix
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=["Malignant", "Benign"],
        cmap="Blues", ax=axes[0]
    )
    axes[0].set_title(f"Confusion Matrix — {model_name}")

    # ROC Curve
    RocCurveDisplay.from_predictions(y_test, y_prob,
                                     name=model_name, ax=axes[1])
    axes[1].plot([0, 1], [0, 1], "--", color="grey", label="Random")
    axes[1].set_title(f"ROC Curve — {model_name} (AUC={auc:.4f})")
    axes[1].legend(loc="lower right")

    plt.tight_layout()
    plt.show()
    return {'model_name': model_name, 'accuracy': acc, 'precision': prec,
            'recall': rec, 'f1': f1, 'auc': auc}

print("evaluate_classification_model() defined.")

In [ ]:
models_to_compare = [
    (LogisticRegression(max_iter=10000, random_state=42), "Logistic Regression"),
    (RandomForestClassifier(n_estimators=100, random_state=42), "Random Forest"),
    (SVC(probability=True, C=1, kernel='rbf', random_state=42), "SVC (rbf)"),
]

results = []
for model, name in models_to_compare:
    r = evaluate_classification_model(model, X_train_sc, X_test_sc, y_train, y_test, name)
    results.append(r)

In [ ]:
# Summary table
summary_df = pd.DataFrame(results).set_index('model_name')
print("\nModel Comparison Summary")
print(summary_df.round(4).to_string())

# Bar chart
summary_df[['accuracy', 'precision', 'recall', 'f1', 'auc']].plot(
    kind='bar', figsize=(10, 5), ylim=(0.9, 1.01), edgecolor='white'
)
plt.title("Model Comparison — Breast Cancer Classification")
plt.ylabel("Score")
plt.xticks(rotation=20, ha='right')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

---

## 14. Practice Exercises

Work through these three exercises to consolidate what you have learned.

### Exercise 1 — Threshold Analysis

Compute precision, recall, and F1 for the Logistic Regression at **threshold = 0.3** and **threshold = 0.7**.
Interpret: which threshold is better for cancer screening? Which is better if you want to minimise false alarms?

*Hint: use `base_lr.predict_proba(X_test_sc)[:, 1]` and compare to a threshold.*

In [ ]:
# Your code here
# y_prob_lr = base_lr.predict_proba(X_test_sc)[:, 1]
# for threshold in [0.3, 0.7]:
#     y_t = (y_prob_lr >= threshold).astype(int)
#     ...

In [ ]:
# --- Solution (expand to reveal) ---
y_prob_lr = base_lr.predict_proba(X_test_sc)[:, 1]

for threshold in [0.3, 0.7]:
    y_t = (y_prob_lr >= threshold).astype(int)
    p = precision_score(y_test, y_t, zero_division=0)
    r = recall_score(y_test, y_t, zero_division=0)
    f = f1_score(y_test, y_t, zero_division=0)
    print(f"Threshold={threshold:.1f} → Precision={p:.4f}  Recall={r:.4f}  F1={f:.4f}")

print()
print("Threshold=0.3: lower bar to call 'benign' → catches more true positives (higher recall).")
print("               More false alarms but fewer missed cancers. Better for screening.")
print("Threshold=0.7: higher bar → fewer false alarms (higher precision) but more missed cancers.")
print("               Better if over-treatment is costly.")

### Exercise 2 — Grid Search on KNN

Use `GridSearchCV` to find the best `k` (number of neighbours) for a `KNeighborsClassifier` on the breast cancer dataset.
Search over `k = 1, 3, 5, 7, 9, 11, 15, 20`. Use 5-fold CV and ROC-AUC scoring.
Plot validation AUC vs k.

*Hint: `from sklearn.neighbors import KNeighborsClassifier`*

In [ ]:
# Your code here
# from sklearn.neighbors import KNeighborsClassifier
# param_grid_knn = {'n_neighbors': [1, 3, 5, 7, 9, 11, 15, 20]}
# knn_grid = GridSearchCV(...)
# knn_grid.fit(X_train_sc, y_train)
# ...

In [ ]:
# --- Solution ---
from sklearn.neighbors import KNeighborsClassifier

param_grid_knn = {'n_neighbors': [1, 3, 5, 7, 9, 11, 15, 20]}
knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1
)
knn_grid.fit(X_train_sc, y_train)

knn_results = pd.DataFrame(knn_grid.cv_results_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(knn_results['param_n_neighbors'], knn_results['mean_test_score'],
        'o-', color='steelblue')
ax.fill_between(knn_results['param_n_neighbors'],
                knn_results['mean_test_score'] - knn_results['std_test_score'],
                knn_results['mean_test_score'] + knn_results['std_test_score'],
                alpha=0.2, color='steelblue')
ax.axvline(knn_grid.best_params_['n_neighbors'], color='tomato', linestyle='--',
           label=f"Best k={knn_grid.best_params_['n_neighbors']}")
ax.set_xlabel('k (n_neighbors)')
ax.set_ylabel('CV ROC-AUC')
ax.set_title('KNN — Validation AUC vs k')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Best k          : {knn_grid.best_params_['n_neighbors']}")
print(f"Best CV AUC     : {knn_grid.best_score_:.4f}")

### Exercise 3 — Learning Curves for Random Forest

Plot the learning curve for a `RandomForestClassifier(n_estimators=50, random_state=42)` on the breast cancer dataset.
Use 5-fold stratified CV and ROC-AUC scoring.
Does the model overfit with small training sets? What happens as data grows?

In [ ]:
# Your code here
# rf = RandomForestClassifier(n_estimators=50, random_state=42)
# train_sizes, train_scores, val_scores = learning_curve(...)
# ...

In [ ]:
# --- Solution ---
rf_lc = RandomForestClassifier(n_estimators=50, random_state=42)

rf_train_sizes, rf_train_scores, rf_val_scores = learning_curve(
    rf_lc, X_sc_full, y_cancer,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)

rf_tr_mean = rf_train_scores.mean(axis=1)
rf_vl_mean = rf_val_scores.mean(axis=1)
rf_vl_std  = rf_val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(rf_train_sizes, rf_tr_mean, 'o-', color='steelblue', label='Training AUC')
ax.plot(rf_train_sizes, rf_vl_mean, 's-', color='tomato',    label='Validation AUC')
ax.fill_between(rf_train_sizes, rf_vl_mean - rf_vl_std, rf_vl_mean + rf_vl_std,
                alpha=0.2, color='tomato')
ax.set_xlabel('Training Set Size')
ax.set_ylabel('ROC-AUC')
ax.set_title('Learning Curve — Random Forest on Breast Cancer')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

gap = rf_tr_mean[-1] - rf_vl_mean[-1]
print(f"Train AUC (full data): {rf_tr_mean[-1]:.4f}")
print(f"Val   AUC (full data): {rf_vl_mean[-1]:.4f}")
print(f"Gap: {gap:.4f}")
print()
print("Observation: Random Forests tend to achieve train AUC ≈ 1.0 due to deep trees,")
print("but validation AUC is high too because bagging reduces variance.")
print("With small training sets the gap is large — classic overfitting regime.")

---

## Summary

| Topic | Key Takeaway |
|-------|--------------|
| Accuracy | Misleading on imbalanced data — always check class distribution first |
| Confusion Matrix | Reveals the *type* of errors; FN vs FP tradeoff depends on business cost |
| Precision / Recall | Precision-recall tradeoff controlled by decision threshold |
| ROC-AUC | Threshold-independent ranking quality; 0.5 = random, 1.0 = perfect |
| PR Curve | Preferred over ROC when the positive class is rare |
| Regression Metrics | MAE for interpretability, RMSE for penalising large errors, R² for explained variance |
| Cross-Validation | 5-fold stratified CV gives a much more reliable estimate than a single split |
| Learning Curves | Diagnose overfitting (large gap) vs underfitting (both scores low) |
| Validation Curves | Find the optimal hyperparameter value visually |
| Grid Search | Exhaustive but expensive; best for small grids |
| Random Search | Sample-efficient for large spaces; use distributions, not fixed grids |
| Evaluation Pipeline | Wrap everything in a reusable function for fair model comparison |

**Next steps:** Apply these techniques to your own dataset in the assignment. Pay special attention to which metric you choose *before* modelling — the metric should reflect what matters in the real problem.